# Finetuning Granite Speech (LoRA)


In [ ]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


## Setup

In [1]:
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
#MODEL_ID = "asr_finetune/checkpoint"

MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [2]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def read_manifest(path):
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)
dev_data = read_manifest(MANIFEST_DEV)

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

10000 train samples, 1000 dev samples
{'audio': '/training-1/mls/mls_english_opus/train/audio/3488/5225/3488_5225_001583.opus', 'text': "it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket joe explained who was the man asked moxlow i thought he looked like john north said montgomery", 'prompt': 'Vocabulary: joe, moxlow, john north, montgomery.'}


## Prompt Construction

In [3]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "can you transcribe the speech into a written format?"

def build_user_prompt(sample):
    prompt = sample.get("prompt")
    if prompt:
        return f"{DEFAULT_PROMPT} Keywords: {prompt}"
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,259,840 || all params: 2,317,401,436 || trainable%: 0.1838


## Collate Function (Audio + Prompt + Target Text)

In [5]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]

    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)

    target_ids = [processor.tokenizer(sample["text"], add_special_tokens=False,
                                      return_tensors="pt").input_ids[0] for sample in batch]
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket joe explained who was the man asked moxlow i thought he looked like john north said montgomery


## Training Configuration

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=1,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [7]:
import jiwer
from tqdm import tqdm

def compute_dev_wer():
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)



In [8]:
print("Dev WER before training:", compute_dev_wer())

  0%|          | 2/1000 [00:45<6:15:58, 22.60s/it]


KeyboardInterrupt: 

## Train

In [11]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    """Print loss and learning rate at every logging step"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,1.487405,0.272765


step 1: {'loss': 16.56620216369629, 'grad_norm': 64.85472106933594, 'learning_rate': 1e-05, 'epoch': 0.0004}
step 10: {'loss': 7.965198092990452, 'grad_norm': 61.66390609741211, 'learning_rate': 9.964e-06, 'epoch': 0.004}
step 20: {'loss': 8.708280944824219, 'grad_norm': 51.186805725097656, 'learning_rate': 9.924e-06, 'epoch': 0.008}
step 30: {'loss': 7.426001739501953, 'grad_norm': 12.234970092773438, 'learning_rate': 9.884e-06, 'epoch': 0.012}
step 40: {'loss': 4.378783416748047, 'grad_norm': 8.522834777832031, 'learning_rate': 9.844000000000001e-06, 'epoch': 0.016}
step 50: {'loss': 5.211905288696289, 'grad_norm': 13.588226318359375, 'learning_rate': 9.804000000000001e-06, 'epoch': 0.02}
step 60: {'loss': 7.833941650390625, 'grad_norm': 54.24129104614258, 'learning_rate': 9.764000000000002e-06, 'epoch': 0.024}
step 70: {'loss': 14.8763671875, 'grad_norm': 120.69051361083984, 'learning_rate': 9.724e-06, 'epoch': 0.028}
step 80: {'loss': 8.35782470703125, 'grad_norm': 11.4412679672241

TrainOutput(global_step=2500, training_loss=1.8097296508789062, metrics={'train_runtime': 3720.255, 'train_samples_per_second': 2.688, 'train_steps_per_second': 0.672, 'total_flos': 2.710951618226515e+16, 'train_loss': 1.8097296508789062, 'epoch': 1.0})

In [13]:
print("Dev WER AFTER training:", compute_dev_wer())

  0%|          | 3/1000 [01:19<7:18:00, 26.36s/it]


KeyboardInterrupt: 

## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [14]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint written to asr_finetune/checkpoint
